<a href="https://colab.research.google.com/github/ekee7/acc_reality_check/blob/main/ACC_Reality_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ACC Reality Check**

**Group Members: Emma Kee (hvh6ng), Kylie Sharp (dhq4mr), and Raffaella Alpaca (bet3at)**

In [1]:
from pathlib import Path

# Create folders for raw data
Path("data/raw/torvik").mkdir(parents=True, exist_ok=True)
Path("data/raw/scoreboard").mkdir(parents=True, exist_ok=True)
Path("data/raw/boxscores").mkdir(parents=True, exist_ok=True)

print("Project folders created")

Project folders created


## **Import Data**

In [2]:
import requests
import pandas as pd

# Download Torvik data
url = "https://raw.githubusercontent.com/ekee7/acc_reality_check/main/2027_team_results.csv"

response = requests.get(url)
response.raise_for_status()

# Save original CSV
file_path = Path("data/raw/torvik/2027_team_results.csv")
file_path.write_bytes(response.content)

# Load CSV into pandas
df_torvik = pd.read_csv(file_path)

# Preserve original copy
df_torvik_original = df_torvik.copy()

print("Dataset downloaded successfully")
print("Dataset shape:", df_torvik.shape)

display(df_torvik.head())

Dataset downloaded successfully
Dataset shape: (365, 45)


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,6-0,124.474683,3,93.804798,3,0.962788,1,...,0.0,0.0,0,0,0.831863,0,2.444834,1,26,67.558509
1,2,Illinois,B10,5-1,126.281293,1,96.602043,16,0.956102,2,...,0.0,0.0,0,0,0.836524,0,0.985492,20,120,73.530920
2,3,Kansas,B12,5-1,122.691516,6,94.294347,4,0.953797,3,...,0.0,0.0,0,0,0.829842,0,1.296530,9,161,72.710877
3,4,Florida,SEC,6-1,124.762015,2,96.285008,13,0.951644,4,...,0.0,0.0,0,0,0.856907,0,2.286053,2,68,68.495192
4,5,Duke,ACC,4-2,119.887207,20,93.240421,1,0.947388,5,...,0.0,0.0,0,0,0.801220,0,-0.309751,88,336,71.564028


In [3]:
# Overview of df_torvik
print("Number of rows:", len(df_torvik))
print("Number of columns:", len(df_torvik.columns))

print("\nColumn names:")
print(df_torvik.columns.tolist())

Number of rows: 365
Number of columns: 45

Column names:
['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


In [4]:
# Filter the dataset to ACC teams
df_acc = df_torvik[df_torvik["conf"] == "ACC"].copy()

# Check the results
print("Number of ACC teams:", len(df_acc))
print("\nACC teams:")
display(df_acc[["team", "conf", "record"]].sort_values("team"))

Number of ACC teams: 18

ACC teams:


,team,conf,record
100,Boston College,ACC,6-0
99,California,ACC,4-2
67,Clemson,ACC,4-1
4,Duke,ACC,4-2
54,Florida St.,ACC,5-1
122,Georgia Tech,ACC,5-1
23,Louisville,ACC,3-1
14,Miami FL,ACC,5-1
32,N.C. State,ACC,5-1
51,North Carolina,ACC,4-1


In [5]:
# Save the ACC teams to a separate CSV file
acc_file_path = Path("data/raw/torvik/acc_teams.csv")
df_acc.to_csv(acc_file_path, index=False)

print("ACC dataset saved successfully")
print("Number of teams saved:", len(df_acc))

ACC dataset saved successfully
Number of teams saved: 18


In [6]:
from datetime import date, timedelta
import json

# ESPN scoreboard API
scoreboard_url = ("https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard")

# Set the date range
start_date = date(2026, 11, 1)
end_date = date(2026, 11, 23)

# Store all scoreboard responses
all_scoreboards = {}

current_date = start_date

while current_date <= end_date:
    date_string = current_date.strftime("%Y%m%d")

    params = {
        "groups": 2,
        "dates": date_string,
        "limit": 200
    }

    response = requests.get(scoreboard_url, params=params)
    response.raise_for_status()

    scoreboard_data = response.json()

    # Save the original ESPN response
    file_path = Path(f"data/raw/scoreboard/scoreboard_{date_string}.json")

    file_path.write_text(json.dumps(scoreboard_data, indent=2))

    all_scoreboards[date_string] = scoreboard_data

    print(
        date_string,
        "- Games found:",
        len(scoreboard_data.get("events", []))
    )

    current_date += timedelta(days=1)

print("\nScoreboard data collection complete")

20261101 - Games found: 1
20261102 - Games found: 13
20261103 - Games found: 4
20261104 - Games found: 1
20261105 - Games found: 7
20261106 - Games found: 9
20261107 - Games found: 1
20261108 - Games found: 3
20261109 - Games found: 5
20261110 - Games found: 10
20261111 - Games found: 2
20261112 - Games found: 0
20261113 - Games found: 11
20261114 - Games found: 3
20261115 - Games found: 3
20261116 - Games found: 2
20261117 - Games found: 10
20261118 - Games found: 3
20261119 - Games found: 5
20261120 - Games found: 9
20261121 - Games found: 3
20261122 - Games found: 2
20261123 - Games found: 3

Scoreboard data collection complete


In [7]:
# Combine games from all saved scoreboard responses
all_games = []

for date_string, scoreboard in all_scoreboards.items():
    for game in scoreboard.get("events", []):
        all_games.append(game)

# Create a DataFrame with basic game information
games_summary = []

for game in all_games:
    competition = game["competitions"][0]

    teams = competition["competitors"]

    games_summary.append({
        "game_id": game["id"],
        "game_name": game["name"],
        "game_date_utc": game["date"],
        "status": competition["status"]["type"]["name"],
        "home_team": next(
            (team["team"]["displayName"] for team in teams
             if team["homeAway"] == "home"), None
        ),
        "away_team": next(
            (team["team"]["displayName"] for team in teams
             if team["homeAway"] == "away"), None
        )
    })

df_games_summary = pd.DataFrame(games_summary)

print("Total games collected:", len(df_games_summary))
print("\nGame statuses:")
print(df_games_summary["status"].value_counts())

display(df_games_summary.head(10))

Total games collected: 110

Game statuses:
status
STATUS_FINAL    110
Name: count, dtype: int64


,game_id,game_name,game_date_utc,status,home_team,away_team
0,401920982,Notre Dame Fighting Irish at Villanova Wildcats,2026-11-01T14:30Z,STATUS_FINAL,Villanova Wildcats,Notre Dame Fighting Irish
1,401906769,Miami Hurricanes at Florida Gators,2026-11-02T23:00Z,STATUS_FINAL,Florida Gators,Miami Hurricanes
2,401909562,Army Black Knights at Duke Blue Devils,2026-11-02T23:30Z,STATUS_FINAL,Duke Blue Devils,Army Black Knights
3,401918947,Chattanooga Mocs at Louisville Cardinals,2026-11-03T02:00Z,STATUS_FINAL,Louisville Cardinals,Chattanooga Mocs
4,401918073,Stetson Hatters at Clemson Tigers,2026-11-03T00:00Z,STATUS_FINAL,Clemson Tigers,Stetson Hatters
5,401913099,Western Carolina Catamounts at North Carolina ...,2026-11-03T00:00Z,STATUS_FINAL,North Carolina Tar Heels,Western Carolina Catamounts
6,401909716,New Haven Chargers at Syracuse Orange,2026-11-03T00:00Z,STATUS_FINAL,Syracuse Orange,New Haven Chargers
7,401909705,Rider Broncs at Pittsburgh Panthers,2026-11-03T00:00Z,STATUS_FINAL,Pittsburgh Panthers,Rider Broncs
8,401918955,Campbell Fighting Camels at NC State Wolfpack,2026-11-03T00:30Z,STATUS_FINAL,NC State Wolfpack,Campbell Fighting Camels
9,401918967,Southern Jaguars at SMU Mustangs,2026-11-03T01:00Z,STATUS_FINAL,SMU Mustangs,Southern Jaguars


In [8]:
# Keep only completed games
df_games_final = df_games_summary[
    df_games_summary["status"] == "STATUS_FINAL"
].copy()

# Check the results
print("Total games collected:", len(df_games_summary))
print("Completed games:", len(df_games_final))
print("Scheduled games excluded:", len(df_games_summary) - len(df_games_final))

display(df_games_final.head())

Total games collected: 110
Completed games: 110
Scheduled games excluded: 0


,game_id,game_name,game_date_utc,status,home_team,away_team
0,401920982,Notre Dame Fighting Irish at Villanova Wildcats,2026-11-01T14:30Z,STATUS_FINAL,Villanova Wildcats,Notre Dame Fighting Irish
1,401906769,Miami Hurricanes at Florida Gators,2026-11-02T23:00Z,STATUS_FINAL,Florida Gators,Miami Hurricanes
2,401909562,Army Black Knights at Duke Blue Devils,2026-11-02T23:30Z,STATUS_FINAL,Duke Blue Devils,Army Black Knights
3,401918947,Chattanooga Mocs at Louisville Cardinals,2026-11-03T02:00Z,STATUS_FINAL,Louisville Cardinals,Chattanooga Mocs
4,401918073,Stetson Hatters at Clemson Tigers,2026-11-03T00:00Z,STATUS_FINAL,Clemson Tigers,Stetson Hatters


In [9]:
# Create the dated Torvik snapshot

# Add the import statement
from datetime import datetime

# Get the current date
snapshot_date = datetime.now().strftime("%Y-%m-%d")

# Create the filename
snapshot_file = f"data/raw/torvik/torvik_{snapshot_date}.csv"

# Save the full dataset - csv. file
df_torvik.to_csv(snapshot_file, index=False)

# Print summary information
print("Snapshot saved:", snapshot_file)
print("Number of teams:", len(df_torvik))
print("Number of columns:", len(df_torvik.columns))

Snapshot saved: data/raw/torvik/torvik_2026-11-26.csv
Number of teams: 365
Number of columns: 45


In [10]:
# ESPN Box Score Data Collection

# Create import statement
import requests

# Select one of the games to inspect the ESPN box score response
game_id = df_games_final.loc[0, "game_id"]

# Establish the URL - changes based on game_id
url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"

# Create response variable
response = requests.get(url)

# Check to see the status code based on the response
print("Status Code:", response.status_code)

# Assign the response to the boxscore_data variable
boxscore_data = response.json()

# Print the avail. section information based on the boxscore_data keys
print("Available sections:", boxscore_data.keys())


Status Code: 200
Available sections: dict_keys(['boxscore', 'format', 'gameInfo', 'leaders', 'broadcasts', 'pickcenter', 'odds', 'againstTheSpread', 'news', 'winprobability', 'plays', 'article', 'videos', 'header', 'wallclockAvailable', 'meta', 'standings'])


In [11]:
# ESPN box score collection - download and save the games in JSON format

# Import libraries
from pathlib import Path
import time
import json
import requests

# Create a folder to hold the raw ESPN box scores
boxscore_folder = Path("/content/data/raw/espn/boxscores")
boxscore_folder.mkdir(parents=True, exist_ok=True)

# Keep track of the successful and unsuccessful downloads from ESPN
successful_downloads = 0  # Initialize at 0
failed_downloads = []

# Download the box score for each completed game
for game_id in df_games_final["game_id"]:

    # Create the URL using the unique game ID
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"

    try:
        # Send a request to ESPN and check for a successful response
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Convert the response into a Python dictionary
        boxscore_data = response.json()

        # Save the raw JSON response using the game ID as the file name
        file_path = boxscore_folder / f"{game_id}.json"

        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(boxscore_data, file, indent=4)

        # Verify that the file exists and is not empty
        if file_path.exists() and file_path.stat().st_size > 0:
            successful_downloads += 1
        else:
            failed_downloads.append((game_id, "File was not saved"))

    except Exception as error:
        # Record the game ID and error if the download fails
        failed_downloads.append((game_id, str(error)))

    # Use a short pause between requests
    time.sleep(0.2)

# Print summary information
print("Successful downloads:", successful_downloads)
print("Failed downloads:", len(failed_downloads))
print("Files saved:", len(list(boxscore_folder.glob("*.json"))))
print("Files saved in:", boxscore_folder)

Successful downloads: 110
Failed downloads: 0
Files saved: 110
Files saved in: /content/data/raw/espn/boxscores


In [12]:
# Verify the downloaded JSON files

# Get a list of all the box score files
boxscore_files = list(boxscore_folder.glob("*.json"))

# Open one of the saved box scores - using read mode
with open(boxscore_files[0], "r", encoding="utf-8") as file:
    sample_boxscore = json.load(file)

# Display the main sections of the JSON response
print("Available sections:", sample_boxscore.keys())

# Inspect the structure of the box score section
print("\nBox score sections:")
print(sample_boxscore["boxscore"].keys())


Available sections: dict_keys(['boxscore', 'format', 'gameInfo', 'leaders', 'broadcasts', 'pickcenter', 'odds', 'againstTheSpread', 'news', 'winprobability', 'plays', 'article', 'videos', 'header', 'wallclockAvailable', 'meta', 'standings'])

Box score sections:
dict_keys(['teams', 'players'])


In [13]:
# Inspect the team information from the previous sample box score

# Print the number of teams that were included in the box score
print("Number of teams:", len(sample_boxscore["boxscore"]["teams"]))

# Print the info avail for the first team
print("\nFirst team information:")
print(sample_boxscore["boxscore"]["teams"][0].keys())

# Print the actual team data
print("\nTeam data:")
print(sample_boxscore["boxscore"]["teams"][0])

Number of teams: 2

First team information:
dict_keys(['team', 'statistics', 'displayOrder', 'homeAway'])

Team data:
{'team': {'id': '56', 'uid': 's:40~l:41~t:56', 'slug': 'stetson-hatters', 'location': 'Stetson', 'name': 'Hatters', 'abbreviation': 'STET', 'displayName': 'Stetson Hatters', 'shortDisplayName': 'Stetson', 'color': '0a5640', 'alternateColor': '56854e', 'logo': 'https://a.espncdn.com/i/teamlogos/ncaa/500/56.png'}, 'statistics': [{'name': 'fieldGoalsMade-fieldGoalsAttempted', 'displayValue': '29-71', 'label': 'FG'}, {'name': 'fieldGoalPct', 'displayValue': '41', 'abbreviation': 'FG%', 'label': 'Field Goal %'}, {'name': 'threePointFieldGoalsMade-threePointFieldGoalsAttempted', 'displayValue': '9-29', 'label': '3PT'}, {'name': 'threePointFieldGoalPct', 'displayValue': '31', 'abbreviation': '3P%', 'label': 'Three Point %'}, {'name': 'freeThrowsMade-freeThrowsAttempted', 'displayValue': '2-6', 'label': 'FT'}, {'name': 'freeThrowPct', 'displayValue': '33', 'abbreviation': 'FT%'

In [14]:
# Inspect the ESPN players section from the sample box score information

# Print the number of teams included in the player section
print("Number of players:", len(sample_boxscore["boxscore"]["players"]))

# Print the available information for the first team
print("\nFirst team information:")
print(sample_boxscore["boxscore"]["players"][0].keys())

# Print the player information
print("\nPlayer information:")
print(sample_boxscore["boxscore"]["players"][0])

Number of players: 2

First team information:
dict_keys(['team', 'statistics', 'displayOrder'])

Player information:
{'team': {'id': '56', 'uid': 's:40~l:41~t:56', 'slug': 'stetson-hatters', 'location': 'Stetson', 'name': 'Hatters', 'abbreviation': 'STET', 'displayName': 'Stetson Hatters', 'shortDisplayName': 'Stetson', 'color': '0a5640', 'alternateColor': '56854e', 'logo': 'https://a.espncdn.com/i/teamlogos/ncaa/500/56.png'}, 'statistics': [{'names': ['MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF'], 'keys': ['minutes', 'points', 'fieldGoalsMade-fieldGoalsAttempted', 'threePointFieldGoalsMade-threePointFieldGoalsAttempted', 'freeThrowsMade-freeThrowsAttempted', 'rebounds', 'assists', 'turnovers', 'steals', 'blocks', 'offensiveRebounds', 'defensiveRebounds', 'fouls'], 'labels': ['MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF'], 'descriptions': ['Minutes', 'Points', 'Field Goals Made/Attempted', '3-Point Fie

In [15]:
# Inspect the individual player

# Fetch the first team in the player section with the 0 index
first_team_players = sample_boxscore["boxscore"]["players"][0]

# Fetch the first player on that team
first_player = first_team_players["statistics"][0]["athletes"][0]

# Print the player's name
print("Player name:", first_player["athlete"]["displayName"])

# Print the player's ID
print("Player ID:", first_player["athlete"]["id"])

# Print if the player is a starter
print("Starter:", first_player["starter"])

# Print the player's statistics
print("Player statistics:", first_player["stats"])

# Print the labels that are associated with the in-game statistics
print("Statistic labels:", first_team_players["statistics"][0]["labels"])

Player name: Makhi Myles
Player ID: 5240393
Starter: True
Player statistics: ['12', '3', '1-2', '1-1', '0-0', '3', '2', '1', '1', '0', '0', '3', '3']
Statistic labels: ['MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF']


**Data Transformation**

Team Name Crosswalk

The ESPN and Bart Torvik datasets use different naming
conventions for some teams. We will create a crosswalk
to match teams across both data sources and validate
that all 18 ACC schools are correctly matched.

In [16]:
#collect unique ESPN teams
espn_teams = {}

for game in all_games:
    competition = game["competitions"][0]

    for competitor in competition["competitors"]:
        team = competitor["team"]

        espn_teams[team["id"]] = team["location"]

#convert to a DataFrame
df_espn_teams = pd.DataFrame(
    espn_teams.items(),
    columns=["espn_id", "espn_name"]
)

print("Number of unique ESPN teams:", len(df_espn_teams))

display(df_espn_teams.head(20))

Number of unique ESPN teams: 116


,espn_id,espn_name
0,222,Villanova
1,87,Notre Dame
2,57,Florida
3,2390,Miami
4,150,Duke
5,349,Army
6,97,Louisville
7,236,Chattanooga
8,228,Clemson
9,56,Stetson


In [17]:
#normalize team names for automatic matching
def normalize_name(name):
    name = name.lower().strip()
    name = name.replace(".", "")
    name = name.replace("-", " ")

    words = name.split()
    words = ["state" if word == "st" else word for word in words]

    return " ".join(words)
#create copies to preserve the original datasets
espn_matching = df_espn_teams.copy()

torvik_matching = df_torvik[["team", "conf"]].copy()

#normalize names in both datasets
espn_matching["normalized_name"] = espn_matching["espn_name"].apply(normalize_name)

torvik_matching["normalized_name"] = torvik_matching["team"].apply(normalize_name)

#match ESPN teams with Torvik teams
df_crosswalk = espn_matching.merge(
    torvik_matching,
    on="normalized_name",
    how="left"
)
#rename the Torvik team column
df_crosswalk = df_crosswalk.rename(columns={"team": "torvik_name"})
display(df_crosswalk.head(20))

,espn_id,espn_name,normalized_name,torvik_name,conf
0,222,Villanova,villanova,Villanova,BE
1,87,Notre Dame,notre dame,Notre Dame,ACC
2,57,Florida,florida,Florida,SEC
3,2390,Miami,miami,NaN,NaN
4,150,Duke,duke,Duke,ACC
5,349,Army,army,Army,Pat
6,97,Louisville,louisville,Louisville,ACC
7,236,Chattanooga,chattanooga,Chattanooga,SC
8,228,Clemson,clemson,Clemson,ACC
9,56,Stetson,stetson,Stetson,ASun


In [18]:
#identify unmatched teams
unmatched_teams = df_crosswalk[
    df_crosswalk["torvik_name"].isna()
]

print("Total ESPN teams:", len(df_espn_teams))

print("Successfully matched teams:",
      df_crosswalk["torvik_name"].notna().sum())

print("Unmatched teams:", len(unmatched_teams))

display(unmatched_teams[["espn_id", "espn_name"]])

Total ESPN teams: 116
Successfully matched teams: 110
Unmatched teams: 6


,espn_id,espn_name
3,2390,Miami
21,2001,Adams State
49,44,American University
74,357,Texas A&M-Corpus Christi
102,2352,Loyola Maryland
114,145,Ole Miss


In [19]:
#finding the correct Torvik names for unmatched teams
display(
    df_torvik[
        df_torvik["team"].str.contains(
            "Miami|Adams|American|Corpus|Loyola|Mississippi|Ole Miss",
            case=False,
            na=False
        )
    ][["team", "conf"]]
)

,team,conf
14,Miami FL,ACC
40,Mississippi,SEC
57,Mississippi St.,SEC
90,Miami OH,MAC
127,Loyola Chicago,A10
133,Loyola Marymount,WCC
159,American,Pat
218,Texas A&M Corpus Chris,Slnd
358,Loyola MD,Pat
361,Mississippi Valley St.,SWAC


In [20]:
#manually correct unmatched team names
manual_matches = {
    "Miami": "Miami FL",
    "American University": "American",
    "Texas A&M-Corpus Christi": "Texas A&M Corpus Chris",
    "Loyola Maryland": "Loyola MD",
    "Ole Miss": "Mississippi"
}

# Apply manual corrections
for espn_name, torvik_name in manual_matches.items():
    df_crosswalk.loc[
        df_crosswalk["espn_name"] == espn_name,
        "torvik_name"
    ] = torvik_name

# Update conference information
conference_map = df_torvik.set_index("team")["conf"]

df_crosswalk["conf"] = df_crosswalk["torvik_name"].map(conference_map)

# Check results after manual corrections
print("Total ESPN teams:", len(df_crosswalk))
print("Matched teams:", df_crosswalk["torvik_name"].notna().sum())
print("Unmatched teams:", df_crosswalk["torvik_name"].isna().sum())

display(df_crosswalk[df_crosswalk["torvik_name"].isna()])

Total ESPN teams: 116
Matched teams: 115
Unmatched teams: 1


,espn_id,espn_name,normalized_name,torvik_name,conf
21,2001,Adams State,adams state,NaN,NaN
